In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
target1 = 'Target1'
target2 = 'Target2'

In [2]:
train = pd.read_parquet('train.parquet')
val = pd.read_parquet('val.parquet')
test = pd.read_parquet('test.parquet')




for col in train.select_dtypes(include='number').columns:
    train[col].fillna(train[col].median(), inplace=True)

for col in train.select_dtypes(include='object').columns:
    train[col].fillna(train[col].mode()[0], inplace=True)




for col in val.select_dtypes(include='number').columns:
    val[col].fillna(val[col].median(), inplace=True)

for col in val.select_dtypes(include='object').columns:
    val[col].fillna(val[col].mode()[0], inplace=True)



for col in test.select_dtypes(include='number').columns:
    test[col].fillna(test[col].median(), inplace=True)

for col in test.select_dtypes(include='object').columns:
    test[col].fillna(test[col].mode()[0], inplace=True)

    


train_new = pd.get_dummies(train, drop_first=True)
val_new = pd.get_dummies(val, drop_first=True)
test_new = pd.get_dummies(test, drop_first=True)

/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_91520/1996482037.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(train[col].median(), inplace=True)
/var/folders/_k/vxqnb8117qb_1tpg9p98mtyc0000gn/T/ipykernel_91520/1996482037.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values 

In [3]:
model = RandomForestClassifier(random_state=42)
model.fit(train_new.drop(columns=['Target1', 'Target2']), train_new['Target1'])

RandomForestClassifier(random_state=42)

In [4]:
print(f"TRAIN : {roc_auc_score(train_new[target1], model.predict_proba(train_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"VAL : {roc_auc_score(val_new[target1], model.predict_proba(val_new.drop(columns=[target1, target2]))[:, 1])}")
print(f"TEST : {roc_auc_score(test_new[target1], model.predict_proba(test_new.drop(columns=[target1, target2]))[:, 1])}")

TRAIN : 0.9999999999923098
VAL : 0.9664495750008246
TEST : 0.9667912407916532


In [5]:
print(f"TRAIN : {f1_score(train_new[target1], model.predict(train_new.drop(columns=[target1, target2])))}")
print(f"VAL : {f1_score(val_new[target1], model.predict(val_new.drop(columns=[target1, target2])))}")
print(f"TEST : {f1_score(test_new[target1], model.predict(test_new.drop(columns=[target1, target2])))}")

TRAIN : 0.9999863032461307
VAL : 0.8143814802590397
TEST : 0.8158034874630349


In [4]:
y_pred = model.predict(test_new.drop(columns=['Target1', 'Target2']))
y_proba = model.predict_proba(test_new.drop(columns=['Target1', 'Target2']))[:, 1]

# Метрики
accuracy = accuracy_score(test_new['Target1'], y_pred)
roc_auc = roc_auc_score(test_new['Target1'], y_proba)

print(f'Accuracy: {accuracy:.3f}')
print(f'ROC AUC: {roc_auc:.3f}')


Accuracy: 0.918
ROC AUC: 0.967


In [5]:
f1 = f1_score(test_new['Target1'], y_pred)
print(f'F1 Score: {f1:.3f}')

F1 Score: 0.816
